# Lab 07 - LSTMs
<a target="_blank" href="https://colab.research.google.com/github/andrew-nash/CS6421-labs-2026/blob/main/Lab07.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


This Lab will look at the biggest flaw of the Vanilla RNN that we saw in the previous lab, and introduce the LSTM that you have seen in the lectures.



In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import torch.nn as nn
from torchinfo import summary
import torch.nn.functional as F

import jax.numpy as jnp
import jax

In [ ]:
jax_key = jax.random.PRNGKey(54321)  

# Vanishing Gradients in the Vanilla RNN

Extending what you have seen in the last lab, and in the lectures, we will implement a basic RNN from scratch in NumPy.



## Data Generation

We will attempt to train a very simple RNN, that predicts the sum of all of the values in the inputted sequence.

Recall the equations defining an RNN

\begin{align}
h_t =&\; \tanh(W_{h} h_{t-1} + W_{x} x_t + b_h) \\
y_t =&\; W_{y} h_t + b_y
\end{align}

In [ ]:
sequence_length = 100
input_dim = 1

In [ ]:
X = jax.random.normal(jax_key, shape=(sequence_length, input_dim))
y_true = jnp.sum(X) # Target: sum of inputs

In [ ]:
input_dim = 1
hidden_dim = 1
output_dim = 1
learning_rate = 0.001

def create_and_train_rnn(Wh_init, title):
    
    # Define the initial weights matrices
    Wh = jnp.array([[Wh_init]])
    Wx = jax.random.normal(jax_key, shape=(hidden_dim, input_dim)) * 0.1
    Wy = jax.random.normal(jax_key,shape=(output_dim, hidden_dim)) * 0.1

    hidden_states = jnp.zeros((sequence_length + 1, hidden_dim))

    # Store gradients
    gradients = []

    # Forward Pass
    for t in range(sequence_length):
        hidden_states=hidden_states.at[t+1].set(jnp.tanh(Wh@hidden_states[t] + Wx@X[t]))

    # Output
    y_pred = Wy@hidden_states[-1]
    loss = (y_pred - y_true)**2

    # Backward Pass (BPTT)
    dLoss_dYpred = 2 * (y_pred - y_true)
    dYpred_dWh = hidden_states[-1]

    # Calculate gradient of hidden state at each step
    dWh = 0
    d_h = dLoss_dYpred * Wy # Initial derivative w.r.t last hidden state

    # Backpropagate through time
    for t in reversed(range(sequence_length)):
        # tanh derivative
        dtanh = 1 - hidden_states[t+1]**2
        d_h = d_h * dtanh
        dWh += d_h@hidden_states[t].T
        d_h = d_h@Wh.T # Propagate back to previous state
        gradients.append(jnp.sum(abs(d_h)))

    return gradients


In [ ]:
# --- Run Scenarios ---
# 1. Exploding Gradient (Wh > 1)
grad_exploding = create_and_train_rnn(Wh_init=1.1, title="Exploding")

# 2. Vanishing Gradient (Wh < 1)
grad_vanishing = create_and_train_rnn(Wh_init=0.9, title="Vanishing")

# --- Plot Results ---
plt.figure(figsize=(10, 5))
plt.plot(grad_exploding, label='Exploding (Wh=1.1)')
plt.plot(grad_vanishing, label='Vanishing (Wh=0.9)')
plt.yscale('log')
plt.xlabel('Steps Backwards (Time)')
plt.ylabel('Gradient Magnitude (Log Scale)')
plt.title('Gradient Behavior in Simple RNN')
plt.legend()
plt.grid(True)
plt.show()

## Analysis

The plot you see illustrates the phenomenon of exploding and vanishing gradients in a simple Recurrent Neural Network (RNN) during backpropagation through time (BPTT).

Here's what each part of the plot means:

**X-axis ('Steps Backwards (Time)')**: This represents how far back in time the gradient is being propagated. For example, '0' would be the gradient at the last time step, '1' one step before that, and so on.

**Y-axis ('Gradient Magnitude (Log Scale)')**: This shows the magnitude (absolute value) of the gradients. The use of a log scale is crucial because it allows us to visualize very large and very small values simultaneously. If it were a linear scale, one type of gradient (exploding or vanishing) would quickly dominate and make the other hard to see.

**Orange Line ('Vanishing (Wh=0.9)')**: This line shows the gradient behavior when the recurrent weight Wh is set to 0.9 (less than 1). As the backpropagation goes further back in time (increasing 'Steps Backwards'), the gradient magnitude decreases rapidly, eventually approaching zero. This is the vanishing gradient problem, where gradients become too small to effectively update the weights for earlier time steps, making it difficult for the RNN to learn long-term dependencies.

**Blue Line ('Exploding (Wh=1.1)')**: This line shows the gradient behavior when the recurrent weight Wh is set to 1.1 (greater than 1). As backpropagation proceeds further back in time, the gradient magnitude increases rapidly and exponentially. This is the exploding gradient problem, where gradients become excessively large, leading to unstable training, large weight updates, and potential numerical overflow.


The key to understanding this plot lies in how the gradient is propagated backwards through time in the train_rnn function. At each step, the gradient d_h is multiplied by two main factors:

The recurrent weight Wh: This is 1.1 for the 'exploding' case and 0.9 for the 'vanishing' case.
The derivative of the tanh activation function: This term is (1 - hidden_states[t+1]**2). The output of tanh is always between -1 and 1. If a hidden_state saturates (meaning it gets very close to 1 or -1), then hidden_states[t+1]**2 gets very close to 1, and consequently, (1 - hidden_states[t+1]**2) becomes very close to zero.
So, the effective multiplicative factor for the gradient at each step is approximately Wh * (1 - hidden_states[t+1]**2).

Here's why the 'exploding' line appears below and also vanishes:

**Exploding Gradient (Wh = 1.1)**: In the forward pass, when Wh is greater than 1, the hidden state values (hidden_states[t]) tend to grow rapidly and can easily saturate the tanh activation function (i.e., become very close to 1 or -1). When this happens, the derivative term (1 - hidden_states[t+1]**2) becomes extremely small (close to 0). This small derivative term dominates the Wh = 1.1 factor, effectively causing the overall multiplicative factor to be much less than 1. As a result, the gradient vanishes even though Wh itself is set for explosion. This also explains why it starts lower: if saturation happens quickly, the very first step backward already experiences this strong dampening effect.

**Vanishing Gradient (Wh = 0.9)**: In this case, Wh is already less than 1, meaning it naturally shrinks gradients. Additionally, because Wh is smaller, the hidden states are less likely to saturate the tanh function as much as in the exploding case. Thus, (1 - hidden_states[t+1]**2) might remain closer to 1, making the overall factor 0.9 * (something close to 1), which still causes gradients to vanish, but perhaps less severely or with a different initial magnitude compared to the heavily saturated 'exploding' case.

*In essence, the plot demonstrates that the tanh activation's saturation can counteract the effect of a large recurrent weight, leading to vanishing gradients even when Wh > 1.*

# LSTM

We will implement an LSTM from scratch in Jax, and compare the gradients as they backpropogaate through time.

Then, we will aplly an LSTM on the language recornition task from last week, and compare performance against the RNN.

The following provides an excellent detailed walkthrough of implementing LSTMs: https://colah.github.io/posts/2015-08-Understanding-LSTMs/

As you have seen in class, the equations defining an LSTM are:

<img src='https://colah.github.io/posts/2015-08-Understanding-LSTMs/img/LSTM3-chain.png' width='750px'/>



\begin{align}
f_t &= \sigma(W_f \cdot [h_{t-1}, x_t] + b_f) && \text{Forget Gate} \\
i_t &= \sigma(W_i \cdot [h_{t-1}, x_t] + b_i) && \text{Input Gate} \\
\tilde{C}_t &= \tanh(W_C \cdot [h_{t-1}, x_t] + b_C) && \text{Candidate Cell State} \\
C_t &= f_t \ast C_{t-1} + i_t \ast \tilde{C}_t && \text{New Cell State} \\
o_t &= \sigma(W_o \cdot [h_{t-1}, x_t] + b_o) && \text{Output Gate} \\
h_t &= o_t \ast \tanh(C_t) && \text{Hidden State}
\end{align}


PyTorch (https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) defines these using slightly different notation:


\begin{array}{ll} \\
    i_t = \sigma(W_{ii} x_t + b_{ii} + W_{hi} h_{t-1} + b_{hi}) \\
    f_t = \sigma(W_{if} x_t + b_{if} + W_{hf} h_{t-1} + b_{hf}) \\
    g_t = \tanh(W_{ig} x_t + b_{ig} + W_{hg} h_{t-1} + b_{hg}) \\
    o_t = \sigma(W_{io} x_t + b_{io} + W_{ho} h_{t-1} + b_{ho}) \\
    c_t = f_t \odot c_{t-1} + i_t \odot g_t \\
    h_t = o_t \odot \tanh(c_t) \\
\end{array}

Observe that each weight and bias $W_f, W_i, W_C, b_f, b_i, b_C$ are in fact implemented each implememnted with two weight matrices or bias vectors repsectively -- one is applied to the input data $X$, the other is applied ot the hidden state $h$. We cannot use the same matrix for both, as we cannot guarantee that the hidden state size (size of the hidden state vector) and input dimension (number of channels) are the same

In the following implementation, we will initialise all of the biases as two to shorten the code. Fixing thems a trivial addition. Biases are necessary, as without these, the Cell states will vanish.

**BONUS** This is particularly critical for the forget gate. The reason we do this, is to "forget" as little as possible in the early stages of traing, and thereby prevent vanishing gradients as much as possible.

In [ ]:
jax_key = jax.random.PRNGKey(42)  

X = jax.random.normal(jax_key, shape=(sequence_length, input_dim))
y_true = jnp.sum(X) # Target: sum of inputs

sequence_length = 100


input_dim = 1
hidden_dim = 1
output_dim = 1
learning_rate = 0.001

def sigmoid(x):
    return 1/(1+jnp.exp(-x))

def create_and_train_lstm(Wi_init, title):
    
    # Define the initial Input Gate Weight as Wi_init
    Wi_x = jnp.full((input_dim, hidden_dim), Wi_init)
    Wi_h = jnp.full((hidden_dim, hidden_dim), Wi_init)
    bi = 2
    # All other weights will be initialsied as random normal matrices
    
    # For the forget Gate
    Wf_x = jax.random.normal(jax_key, shape=(input_dim, hidden_dim)) * 0.1 
    Wf_h = jax.random.normal(jax_key, shape=(hidden_dim, hidden_dim)) * 0.1 
    bf = 2
    # For the Candidate Cell State
    Wcc_x = jax.random.normal(jax_key, shape=(input_dim, hidden_dim)) * 0.1
    Wcc_h = jax.random.normal(jax_key, shape=(hidden_dim, hidden_dim)) * 0.1   
    
    # For the Output Gate
    Wo_x = jax.random.normal(jax_key, shape=(input_dim, hidden_dim)) * 0.1
    Wo_h = jax.random.normal(jax_key, shape=(hidden_dim, hidden_dim)) * 0.1  
    bo = 2
    
    # Final Linear Prediction Layer
    Wy = jax.random.normal(jax_key,shape=(hidden_dim, output_dim)) * 0.1
    
    # all hidden states, initially 0s, updated as we genberate them
    hidden_states = jnp.zeros((sequence_length + 1, hidden_dim))
    cell_states = jnp.zeros((sequence_length + 1, hidden_dim))
    # Store gradients
    gradients = [[],[]]
    
    # Backprop requires saving the gate outputs
    i_gates, f_gates, o_gates, cc_gates = [], [], [], []

    # Forward Pass
    for t in range(sequence_length):
        i_t = sigmoid(X[t]@Wi_x + hidden_states[t]@Wi_h + bi)
        f_t = sigmoid(X[t]@Wf_x + hidden_states[t]@Wf_h + bf)
        cc_t = jnp.tanh(X[t]@Wcc_x + hidden_states[t]@Wcc_h)
        
        # NB - note that these are element-wise multiplication
        # not matrix multiplication
        C_next_t = f_t*cell_states[t]+i_t*cc_t
                
        o_t = sigmoid(X[t]@Wo_x+hidden_states[t]@Wo_h+bo)
        h_t = o_t*jnp.tanh(C_next_t)
        
        cell_states=cell_states.at[t+1].set(C_next_t)
        hidden_states=hidden_states.at[t+1].set(h_t)
        
        i_gates.append(i_t)
        f_gates.append(f_t)
        o_gates.append(o_t)
        cc_gates.append(cc_t)
        
    # Output
    y_pred = hidden_states[-1]@Wy
    loss = (y_pred - y_true)**2

    # Backward Pass (BPTT)
    dLoss_dYpred = 2 * (y_pred - y_true)
    dYpred_dWh = hidden_states[-1]

    # Calculate gradient of hidden state at each step
    dWh = 0
    d_h = Wy@dLoss_dYpred # Initial derivative w.r.t last hidden state
    
    d_C = jnp.zeros(hidden_dim) # Initial derivative w.r.t last Cell state
    # Backpropagate through time
    for t in reversed(range(sequence_length)):
        # 1. Gradient of h_t
        # d_h is passed from the future + the prediction layer
        
        # 2. Gradient of the Output Gate
        do = d_h * jnp.tanh(cell_states[t+1])
        do_input = do * o_gates[t] * (1 - o_gates[t]) # Sigmoid deriv
        
        # 3. Gradient of the Cell State (c_t)
        # Includes flow from h_t and the future cell state d_C
        dc = d_h * o_gates[t] * (1 - jnp.tanh(cell_states[t+1])**2) + d_C
        
        # 4. Gradient of the Input Gate (The one you are testing)
        di = dc * cc_gates[t]
        di_input = di * i_gates[t] * (1 - i_gates[t]) # Sigmoid deriv
        
        # 5. Gradient of the Forget Gate
        df = dc * cell_states[t]
        df_input = df * f_gates[t] * (1 - f_gates[t]) # Sigmoid deriv
        
        # 6. Gradient of the Candidate Cell
        dcc = dc * i_gates[t]
        dcc_input = dcc * (1 - cc_gates[t]**2) # Tanh deriv
        
        # We could also get the gradient w.r.t the input Gate
        # as follows:
        # dL/dWi_x = di_input * X[t]
        # g_Wi_x = X[t].T @ di_input
        # but we don't actually need it in this instance

        # 7. Pass gradients back to previous timestep (t-1)
        # dh_prev = sum of (gate_input_deriv @ Weight_h.T)
        d_h = (di_input @ Wi_h.T + df_input @ Wf_h.T + 
                   do_input @ Wo_h.T + dcc_input @ Wcc_h.T)
        d_C = dc * f_gates[t] # Cell state gradient flows through forget gate
        
        
        gradients[0].append(jnp.sum(jnp.abs(d_h)))
        gradients[1].append(jnp.sum(jnp.abs(d_C)))
        
        
    return np.array(gradients) # Return as an np array for simplicity

In [ ]:
# --- Run Scenarios ---
# 1. Exploding Gradient (Wh > 1)
grad_exploding = create_and_train_lstm(Wi_init=1.1, title="Exploding")

# 2. Vanishing Gradient (Wh < 1)
grad_vanishing = create_and_train_lstm(Wi_init=0.9, title="Vanishing")

# --- Plot Results ---
fig, ax = plt.subplots(2,1,figsize=(10, 10))
ax[0].plot(grad_exploding[0,:], label='Exploding (Wh=1.1)')
ax[0].plot(grad_vanishing[0,:], label='Vanishing (Wh=0.9)')
ax[0].set_yscale('log')
ax[0].set_xlabel('Steps Backwards (Time)')
ax[0].set_ylabel('Gradient Magnitude (Log Scale)')
ax[0].set_title('Hidden State Gradient Behavior in LSTM')
ax[0].legend()
ax[0].grid(True)
ax[1].plot(grad_exploding[1,:], label='Exploding (Wh=1.1)')
ax[1].plot(grad_vanishing[1,:], label='Vanishing (Wh=0.9)')
ax[1].set_yscale('log')
ax[1].set_xlabel('Steps Backwards (Time)')
ax[1].set_ylabel('Gradient Magnitude (Log Scale)')
ax[1].set_title('Cell State Gradient Behavior in LSTM')
ax[1].legend()
ax[1].grid(True)
plt.show()

Notice that gradients do still evntually vanish. However, they do so more consistently, and at a slower rate than the RNN.

## LSTM in PyTorch

In [ ]:
!wget https://download.pytorch.org/tutorial/data.zip
!unzip data.zip -o

# Check if CUDA is available
device = torch.device('cpu')
if torch.cuda.is_available():
    device = torch.device('cuda')

torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

import string
import unicodedata

# We can use "_" to represent an out-of-vocabulary character, that is, any character we are not handling in our model
allowed_characters = string.ascii_letters + " .,;'" + "_"
n_letters = len(allowed_characters)

# Turn a Unicode string to plain ASCII, thanks to https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in allowed_characters
    )

# Find letter index from all_letters, e.g. "a" = 0
def letterToIndex(letter):
    # return our out-of-vocabulary character if we encounter a letter unknown to our model
    if letter not in allowed_characters:
        return allowed_characters.find("_")
    else:
        return allowed_characters.find(letter)

# Turn a line into a <line_length x 1 x n_letters>,
# or an array of one-hot letter vectors
def lineToTensor(line):
    tensor = torch.zeros(len(line), 1, n_letters)
    for li, letter in enumerate(line):
        tensor[li][0][letterToIndex(letter)] = 1
    return tensor

from io import open
import glob
import os
import time

from torch.utils.data import Dataset

class NamesDataset(Dataset):

    def __init__(self, data_dir):
        self.data_dir = data_dir #for provenance of the dataset
        self.load_time = time.localtime #for provenance of the dataset
        labels_set = set() #set of all classes

        self.data = []
        self.data_tensors = []
        self.labels = []
        self.labels_tensors = []

        #read all the ``.txt`` files in the specified directory
        text_files = glob.glob(os.path.join(data_dir, '*.txt'))
        for filename in text_files:
            label = os.path.splitext(os.path.basename(filename))[0]
            labels_set.add(label)
            lines = open(filename, encoding='utf-8').read().strip().split('\n')
            for name in lines:
                self.data.append(name)
                self.data_tensors.append(lineToTensor(name))
                self.labels.append(label)

        #Cache the tensor representation of the labels
        self.labels_uniq = list(labels_set)
        for idx in range(len(self.labels)):
            temp_tensor = torch.tensor([self.labels_uniq.index(self.labels[idx])], dtype=torch.long)
            self.labels_tensors.append(temp_tensor)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]
        data_label = self.labels[idx]
        data_tensor = self.data_tensors[idx]
        label_tensor = self.labels_tensors[idx]

        return label_tensor, data_tensor, data_label, data_item
    
alldata = NamesDataset("data/names")
print(f"loaded {len(alldata)} items of data")
print(f"example = {alldata[0]}")

train_set, test_set = torch.utils.data.random_split(alldata, [.85, .15], generator=torch.Generator(device=device).manual_seed(2024))

print(f"train examples = {len(train_set)}, validation examples = {len(test_set)}")

### Training Loop

In [ ]:
import random
import numpy as np

def train(model, training_data, n_epoch = 10, n_batch_size = 64, report_every = 50, learning_rate = 0.2, criterion = nn.CrossEntropyLoss()):
    """
    Learn on a batch of training_data for a specified number of iterations and reporting thresholds
    """
    # Keep track of losses for plotting
    current_loss = 0
    all_losses = []
    model.train()
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

    start = time.time()
    print(f"training on data set with n = {len(training_data)}")

    for iter in range(1, n_epoch + 1):
        model.zero_grad() # clear the gradients

        # create some minibatches
        # we cannot use dataloaders because each of our names is a different length
        batches = list(range(len(training_data)))
        random.shuffle(batches)
        batches = np.array_split(batches, len(batches) //n_batch_size )

        for idx, batch in enumerate(batches):
            batch_loss = 0
            for i in batch: #for each example in this batch
                (label_tensor, text_tensor, label, text) = training_data[i]
                one_hot_label = torch.zeros((1,len(alldata.labels_uniq)))
                one_hot_label[0,label_tensor[0]]=1
                output = model.forward(text_tensor)
                loss = criterion(output, one_hot_label)
                batch_loss += loss

            # optimize parameters
            batch_loss.backward()
            
            # HERE IS CLIPPING BY NORM!
            # In this case, the norm over all parameters is clipped at 3 
            nn.utils.clip_grad_norm_(model.parameters(), 3)
            optimizer.step()
            optimizer.zero_grad()

            current_loss += batch_loss.item() / len(batch)

        all_losses.append(current_loss / len(batches) )
        if iter % report_every == 0:
            print(f"{iter} ({iter / n_epoch:.0%}): \t average batch loss = {all_losses[-1]}")
        current_loss = 0


    return all_losses

In [ ]:
def label_from_output(output, output_labels):
    top_n, top_i = output.topk(1)
    label_i = top_i[0].item()
    return output_labels[label_i], label_i



In [ ]:
def evaluate(rnn, testing_data, classes):
    confusion = torch.zeros(len(classes), len(classes))

    rnn.eval() #set to eval mode
    with torch.no_grad(): # do not record the gradients during eval phase
        for i in range(len(testing_data)):
            (label_tensor, text_tensor, label, text) = testing_data[i]
            output = rnn(text_tensor)
            guess, guess_i = label_from_output(output, classes)
            label_i = classes.index(label)
            confusion[label_i][guess_i] += 1

    # Normalize by dividing every row by its sum
    for i in range(len(classes)):
        denom = confusion[i].sum()
        if denom > 0:
            confusion[i] = confusion[i] / denom

    # Set up plot
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(confusion.cpu().numpy()) #numpy uses cpu here so we need to use a cpu version
    fig.colorbar(cax)
    
    # Set up axes
    ax.set_xticks(np.arange(len(classes)), labels=classes, rotation=90)
    ax.set_yticks(np.arange(len(classes)), labels=classes)

    # Force label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    # sphinx_gallery_thumbnail_number = 2
    plt.show()




### Baseline RNN

In [ ]:

class CharRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharRNN, self).__init__()

        self.rnn = nn.RNN(input_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        rnn_out, hidden = self.rnn(line_tensor)
        output = self.h2o(hidden[0])
        output = self.softmax(output)

        return output

In [ ]:
n_hidden = 128
rnn = CharRNN(n_letters, n_hidden, len(alldata.labels_uniq))
print(rnn)

In [ ]:
start = time.time()
all_losses = train(rnn, train_set, n_epoch=27, learning_rate=0.15, report_every=5)
end = time.time()
print(f"training took {end-start}s")

In [ ]:
input = lineToTensor('Albert')
output = rnn(input) #this is equivalent to ``output = rnn.forward(input)``
print(output)
print(label_from_output(output, alldata.labels_uniq))

In [ ]:
plt.figure()
plt.plot(all_losses)
plt.show()

In [ ]:

evaluate(rnn, test_set, classes=alldata.labels_uniq)

### Implementation
There is almost entirely no difference!

In [ ]:
class CharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(CharLSTM, self).__init__()

        self.lstm = nn.LSTM(input_size, hidden_size)
        self.h2o = nn.Linear(hidden_size, output_size)
        self.softmax = nn.LogSoftmax(dim=1)

    def forward(self, line_tensor):
        rnn_out, (hidden, cell) = self.lstm(line_tensor)
        output = self.h2o(hidden[0])
        output = self.softmax(output)

        return output
    

In [ ]:
n_hidden = 128
lstm = CharLSTM(n_letters, n_hidden, len(alldata.labels_uniq))
print(lstm)

In [ ]:
summary(lstm)

In [ ]:
start = time.time()
all_losses = train(lstm, train_set, n_epoch=27, learning_rate=0.15, report_every=5)
end = time.time()
print(f"training took {end-start}s")

In [ ]:
input = lineToTensor('Albert')
output = lstm(input) #this is equivalent to ``output = rnn.forward(input)``
print(output)
print(label_from_output(output, alldata.labels_uniq))

In [ ]:
plt.figure()
plt.plot(all_losses)
plt.show()

In [ ]:
evaluate(lstm, test_set, classes=alldata.labels_uniq)